# Modelling — WOE Scorecard
## Credit Risk Scorecard Project

This notebook builds a traditional credit scorecard using:
- Weight of Evidence (WOE) encoding
- Information Value (IV) for feature selection
- Logistic Regression for scoring

This is the interpretable model that banks use in practice because
regulators require explainability. We'll compare it against LightGBM
in the next notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
import warnings
import os
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)

print("Libraries loaded successfully")

In [ ]:
# Load the cleaned dataset from data preparation
app = pd.read_csv('data/processed/app_clean.csv')
print(f"Shape: {app.shape}")
print(f"Default rate: {app['TARGET'].mean():.2%}")

In [ ]:
# Separate features and target
X = app.drop(columns=['TARGET', 'SK_ID_CURR'])
y = app['TARGET']

# Random stratified 80/20 split
# stratify=y ensures both splits have same 8.07% default rate
# Note: a true time-based split would be more rigorous for production
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

# Impute using TRAIN medians only, apply the same values to test
train_medians = X_train.median()
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Train default rate: {y_train.mean():.2%}")
print(f"Test default rate: {y_test.mean():.2%}")

In [ ]:
import scorecardpy as sc

# WOE binning on training data only
# We use a subset of features to keep it manageable
# Focus on key features we engineered plus EXT_SOURCE columns

selected_features = [
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'DAYS_BIRTH', 'DAYS_EMPLOYED', 'AGE_YEARS',
    'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO',
    'CREDIT_GOODS_RATIO', 'INCOME_PER_PERSON',
    'EMPLOYED_TO_AGE_RATIO',
    'AMT_CREDIT', 'AMT_INCOME_TOTAL', 'AMT_ANNUITY',
    'BUREAU_LOAN_COUNT', 'BUREAU_DEBT_MEAN',
    'BUREAU_ACTIVE_COUNT', 'BUREAU_OVERDUE_MEAN',
    'PREV_APP_COUNT', 'PREV_REFUSED_COUNT',
    'PREV_APPROVED_COUNT',
    'TARGET'
]

# Create training dataframe with selected features
train_woe = X_train.copy()
train_woe['TARGET'] = y_train.values
train_woe = train_woe[selected_features]

print(f"Training data for WOE: {train_woe.shape}")

# Calculate WOE bins
bins = sc.woebin(train_woe, y='TARGET')
print("WOE binning complete")

In [ ]:
# IV must come from the binned data (woebin), not sc.iv(), which treats
# every unique value of a continuous variable as its own group.
iv_values = (
    pd.concat(bins.values())[['variable', 'total_iv']]
      .drop_duplicates()
      .rename(columns={'total_iv': 'info_value'})
      .sort_values('info_value', ascending=False)
      .reset_index(drop=True)
)
print("Information Value by feature (from binned data):")
print(iv_values.to_string())

### Note on IV calculation
An earlier version used `sc.iv()`, which treats every unique value as its own
group. For continuous variables (e.g. CREDIT_INCOME_RATIO has 42,625 unique
values in 246,008 rows) this memorises the data and inflates IV (1.07 vs the
true 0.011). IV is now read from the `woebin` bins (`total_iv`). The strongest
predictors are EXT_SOURCE_3 (0.30), EXT_SOURCE_2 (0.30) and EXT_SOURCE_1 (0.14).
CREDIT_INCOME_RATIO (0.011) is weak, consistent with its bin default rates
rising only from 6.6% to 8.9%.

In [ ]:
# 1) Confirm what sc.iv() actually does
help(sc.iv)

# 2) How many distinct values does the "high-IV" variable have?
print(train_woe['CREDIT_INCOME_RATIO'].nunique(), "unique values in", len(train_woe), "rows")

# 3) IV from the binned data (the correct source)
iv_from_bins = (
    pd.concat(bins.values())[['variable', 'total_iv']]
      .drop_duplicates()
      .sort_values('total_iv', ascending=False)
      .reset_index(drop=True)
)
print(iv_from_bins.to_string())

In [ ]:
# Keep features with IV > 0.02, and drop DAYS_BIRTH (duplicate of AGE_YEARS)
iv_filtered = iv_values[iv_values['info_value'] > 0.02]
good_features = [f for f in iv_filtered['variable'].tolist() if f != 'DAYS_BIRTH']

print(f"Features kept: {len(good_features)}")
print(good_features)

# Build the test set with the same raw columns as train_woe
test_woe = X_test.copy()
test_woe['TARGET'] = y_test.values
test_woe = test_woe[selected_features]

# Rebuild train/test using ONLY the selected features (+ TARGET)
cols = good_features + ['TARGET']
train_sel = train_woe[cols]
test_sel  = test_woe[cols]

# Re-bin on the selected features and transform both sets with the same bins
bins_sel = {k: v for k, v in bins.items() if k in good_features}
train_woe_transformed = sc.woebin_ply(train_sel, bins_sel)
test_woe_transformed  = sc.woebin_ply(test_sel,  bins_sel)

print(train_woe_transformed.shape, test_woe_transformed.shape)

In [ ]:
# Prepare final train and test sets
feature_cols = [col for col in train_woe_transformed.columns 
                if col != 'TARGET']

X_train_woe = train_woe_transformed[feature_cols]
y_train_woe = train_woe_transformed['TARGET']

X_test_woe = test_woe_transformed[feature_cols]
y_test_woe = test_woe_transformed['TARGET']


In [ ]:
# Train logistic regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_woe, y_train_woe)


In [ ]:
# Get predicted probabilities
train_pred = lr.predict_proba(X_train_woe)[:, 1]
test_pred = lr.predict_proba(X_test_woe)[:, 1]


In [ ]:
# AUC scores
train_auc = roc_auc_score(y_train_woe, train_pred)
test_auc = roc_auc_score(y_test_woe, test_pred)

print(f"Train AUC: {train_auc:.4f}")
print(f"Test AUC:  {test_auc:.4f}")
print(f"Difference: {train_auc - test_auc:.4f}")

In [ ]:
# Convert logistic regression to scorecard points
scorecard = sc.scorecard(bins, lr, feature_cols)

# Apply scorecard to get credit scores
train_score = sc.scorecard_ply(train_woe, scorecard, print_step=0)
test_score = sc.scorecard_ply(test_woe, scorecard, print_step=0)

print("Scorecard built successfully")
print(f"\nTrain score distribution:")
print(train_score['score'].describe().round(0))
print(f"\nTest score distribution:")
print(test_score['score'].describe().round(0))

In [ ]:
os.makedirs('outputs', exist_ok=True)

# Add target to score dataframes
train_score['TARGET'] = y_train_woe.values
test_score['TARGET'] = y_test_woe.values

# Plot score distribution by default status
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(train_score[train_score['TARGET']==0]['score'], 
        bins=50, alpha=0.6, color='steelblue', label='Non-defaulter (0)')
ax.hist(train_score[train_score['TARGET']==1]['score'], 
        bins=50, alpha=0.6, color='crimson', label='Defaulter (1)')

ax.axvline(x=588, color='black', linestyle='--', label='Mean score (588)')
ax.set_title('Credit Score Distribution by Default Status')
ax.set_xlabel('Credit Score')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/03_score_distribution.png', dpi=150)
plt.show()

print("Chart saved")

In [ ]:
# Verify CREDIT_INCOME_RATIO bins are sensible (not leakage)
sc.woebin_plot(bins['CREDIT_INCOME_RATIO'])

# Fix x-axis label formatting
fig = plt.gcf()
ax = plt.gca()
ax.set_xticklabels(['<1.2', '1.2-1.6', '1.6-2.2', '2.2-5.0', '>5.0'], 
                   rotation=0)
ax.set_xlabel('Credit to Income Ratio')
plt.tight_layout()
plt.savefig('outputs/03_credit_income_ratio_bins.png', dpi=150)
plt.show()

# Print the bin details
print(bins['CREDIT_INCOME_RATIO'][['bin', 'count', 'count_distr', 
                                     'bad', 'badprob', 'woe']].to_string())

### Leakage Check: CREDIT_INCOME_RATIO
Verifying that the engineered ratio features do not encode the target.
Bin analysis confirms default rates rise modestly with the ratio 
(6.6% to 8.9%), consistent with economic intuition — financial stress 
slightly increases default risk. No leakage present. The relationship 
is weak (IV 0.011) so the feature contributes little signal but is retained.

In [ ]:
from sklearn.metrics import roc_curve

# Calculate KS and Gini for the scorecard (to match LightGBM evaluation)
def calculate_ks(y_true, y_pred):
    fpr, tpr, thresholds = roc_curve(y_true, y_pred)
    return max(tpr - fpr)

# Use the test predictions from the logistic regression scorecard
scorecard_ks = calculate_ks(y_test_woe, test_pred)
scorecard_gini = 2 * roc_auc_score(y_test_woe, test_pred) - 1

print(f"Scorecard KS:   {scorecard_ks:.4f}")
print(f"Scorecard Gini: {scorecard_gini:.4f}")
print(f"Scorecard AUC:  {roc_auc_score(y_test_woe, test_pred):.4f}")

In [ ]:
chk = pd.read_csv('data/processed/app_clean.csv')
print("shape:", chk.shape)
print("NaN count in file:", chk.isnull().sum().sum())
print("NaN in X_train before fillna should be > 0; file NaN count tells us which version this is")